# 03. Modelos Base y Evaluacion

Este notebook toma el dataset maestro generado en Notebook 02 y construye:
- baseline multiclase
- modelos base de clasificacion
- comparacion probabilistica por log loss
- modelo Poisson para distribucion de goles

In [96]:
import os
from pathlib import Path

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

modelingBasePath = Path("../data/intermediate/modeling_base_dataset.csv")
print("Modeling base path:", modelingBasePath)

Modeling base path: ..\data\intermediate\modeling_base_dataset.csv


## 1. Carga del dataset maestro

In [97]:
df = pd.read_csv(modelingBasePath)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date", kind="mergesort").reset_index(drop=True)

print("Shape df:", df.shape)
df.head()

Shape df: (25013, 72)


,date,homeTeam,awayTeam,tournament,neutral,matchMonth,matchYear,homeScore,awayScore,target,targetEncoded,homeElo,awayElo,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,matchesPlayedDiff,homeWinRate,awayWinRate,winRateDiff,homeDrawRate,awayDrawRate,drawRateDiff,homeLossRate,awayLossRate,lossRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForAvgDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstAvgDiff,homeGoalDiffAvg,awayGoalDiffAvg,goalDiffAvgDiff,homeLast5WinRate,awayLast5WinRate,last5WinRateDiff,homeLast5PointsAvg,awayLast5PointsAvg,last5PointsDiff,homeLast5GoalsForAvg,awayLast5GoalsForAvg,last5GoalsForDiff,homeLast5GoalsAgainstAvg,awayLast5GoalsAgainstAvg,last5GoalsAgainstDiff,homeLast5GoalDiffAvg,awayLast5GoalDiffAvg,last5GoalDiffDiff,homeLast10PointsAvg,awayLast10PointsAvg,last10PointsDiff,homeLast10GoalsForAvg,awayLast10GoalsForAvg,last10GoalsForDiff,homeLast10GoalsAgainstAvg,awayLast10GoalsAgainstAvg,last10GoalsAgainstDiff,homeDaysSinceLastMatch,awayDaysSinceLastMatch,daysSinceLastMatchDiff,homeH2HWinRate,awayH2HWinRate,h2hDrawRate,tournamentCat_continental,tournamentCat_friendly,tournamentCat_other,tournamentCat_qualification,tournamentCat_worldCup
0,2000-01-04,Egypt,Togo,Friendly,0,1,2000,2,1,win,2,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
1,2000-01-07,Tunisia,Togo,Friendly,0,1,2000,7,0,win,2,1500.0,1491.849325,8.150675,8.150675,0.603744,0,1,-1,0.391509,0.000000,0.015171,0.23913,0.000000,0.0,0.352941,1.000000,-0.017047,1.392857,1.000000,0.043808,1.225806,2.000000,-0.047008,0.185185,-1.000000,0.09632,0.4,0.0,0.0,1.4,0.0,0.0,1.2,1.0,0.0,1.2,2.0,0.0,0.0,-1.0,0.0,1.4,0.0,0.0,1.3,1.0,0.0,1.2,2.0,0.0,12.0,3.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
2,2000-01-08,Trinidad and Tobago,Canada,Friendly,0,1,2000,0,0,draw,1,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
3,2000-01-09,Burkina Faso,Gabon,Friendly,0,1,2000,1,1,draw,1,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
4,2000-01-09,Guatemala,Armenia,Friendly,1,1,2000,1,1,draw,1,1500.0,1500.000000,0.000000,0.000000,0.500000,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0


## 2. Validacion y definicion de features

In [98]:
requiredColumns = [
    "date",
    "homeTeam",
    "awayTeam",
    "tournament",
    "homeScore",
    "awayScore",
    "target",
    "targetEncoded",
]

missingColumns = [col for col in requiredColumns if col not in df.columns]
assert not missingColumns, f"Faltan columnas requeridas: {missingColumns}"
assert df[requiredColumns].isnull().sum().sum() == 0, "Hay nulos en columnas criticas"

nonFeatureColumns = [
    "date",
    "homeTeam",
    "awayTeam",
    "tournament",
    "homeScore",
    "awayScore",
    "target",
    "targetEncoded",
]
featureColumns = [col for col in df.columns if col not in nonFeatureColumns]

X = df[featureColumns].copy()
y = df["targetEncoded"].copy()

nonNumericFeatureColumns = X.select_dtypes(exclude=[np.number]).columns.tolist()
assert not nonNumericFeatureColumns, f"Hay features no numericas: {nonNumericFeatureColumns}"

print("Numero de features:", len(featureColumns))
print("Distribucion target:")
print(df["target"].value_counts(normalize=True).round(4))

Numero de features: 64
Distribucion target:
target
win     0.4811
loss    0.2859
draw    0.2329
Name: proportion, dtype: float64


## 3. Split temporal

In [99]:
trainEnd = int(len(df) * 0.70)
valEnd = trainEnd + int(len(df) * 0.15)

trainDf = df.iloc[:trainEnd].copy()
valDf = df.iloc[trainEnd:valEnd].copy()
testDf = df.iloc[valEnd:].copy()

X_train = trainDf[featureColumns].copy()
X_val = valDf[featureColumns].copy()
X_test = testDf[featureColumns].copy()

y_train = trainDf["targetEncoded"].copy()
y_val = valDf["targetEncoded"].copy()
y_test = testDf["targetEncoded"].copy()

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (17509, 64)
Val: (3751, 64)
Test: (3753, 64)


## 4. Baseline de referencia

In [100]:
majorityClass = y_train.value_counts().idxmax()
y_val_pred_base = np.full_like(y_val, majorityClass)

print("BASELINE (VAL)")
print("Accuracy:", accuracy_score(y_val, y_val_pred_base))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred_base))
print("F1:", f1_score(y_val, y_val_pred_base, average="macro"))

BASELINE (VAL)
Accuracy: 0.48067182084777393
Balanced Accuracy: 0.3333333333333333
F1: 0.21642059776737488


## 5. Modelos base de clasificacion

In [101]:
classifierScaler = StandardScaler()
X_train_scaled = classifierScaler.fit_transform(X_train)
X_val_scaled = classifierScaler.transform(X_val)
X_test_scaled = classifierScaler.transform(X_test)

logModel = LogisticRegression(max_iter=2000)
rfModel = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1,
    class_weight="balanced_subsample",
)

logModel.fit(X_train_scaled, y_train)
rfModel.fit(X_train, y_train)


def metricRow(modelName, yTrue, yPred, yProba):
    return {
        "model": modelName,
        "accuracy": accuracy_score(yTrue, yPred),
        "balancedAccuracy": balanced_accuracy_score(yTrue, yPred),
        "f1": f1_score(yTrue, yPred, average="macro"),
        "logLoss": log_loss(yTrue, yProba, labels=[0, 1, 2]),
    }


y_val_pred_log = logModel.predict(X_val_scaled)
y_val_proba_log = logModel.predict_proba(X_val_scaled)
y_val_pred_rf = rfModel.predict(X_val)
y_val_proba_rf = rfModel.predict_proba(X_val)

validationResults = pd.DataFrame(
    [
        metricRow("LogisticRegression", y_val, y_val_pred_log, y_val_proba_log),
        metricRow("RandomForest", y_val, y_val_pred_rf, y_val_proba_rf),
    ]
).sort_values(["logLoss", "balancedAccuracy", "f1"], ascending=[True, False, False]).reset_index(drop=True)

validationResults

,model,accuracy,balancedAccuracy,f1,logLoss
0,LogisticRegression,0.605705,0.510660,0.470925,0.856007
1,RandomForest,0.580912,0.525775,0.518648,0.889813


## 6. Seleccion automatica del mejor clasificador

In [102]:
bestClassifierName = validationResults.iloc[0]["model"]

classifierSpecs = {
    "LogisticRegression": {
        "model": logModel,
        "X_test": X_test_scaled,
    },
    "RandomForest": {
        "model": rfModel,
        "X_test": X_test,
    },
}

bestClassifier = classifierSpecs[bestClassifierName]["model"]
bestClassifierXTest = classifierSpecs[bestClassifierName]["X_test"]

y_test_pred_best = bestClassifier.predict(bestClassifierXTest)
y_test_proba_best = bestClassifier.predict_proba(bestClassifierXTest)

testBestResults = pd.DataFrame(
    [metricRow(bestClassifierName, y_test, y_test_pred_best, y_test_proba_best)]
)

print("Best classifier selected:", bestClassifierName)
testBestResults

Best classifier selected: LogisticRegression


,model,accuracy,balancedAccuracy,f1,logLoss
0,LogisticRegression,0.60858,0.516203,0.476375,0.873293


## 7. Evaluacion final del mejor clasificador

In [103]:
print("TEST RESULTS -", bestClassifierName)
print("Accuracy:", accuracy_score(y_test, y_test_pred_best))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_test_pred_best))
print("F1:", f1_score(y_test, y_test_pred_best, average="macro"))
print("LogLoss:", log_loss(y_test, y_test_proba_best, labels=[0, 1, 2]))
print()
print(classification_report(y_test, y_test_pred_best, zero_division=0))

TEST RESULTS - LogisticRegression
Accuracy: 0.6085798028244072
Balanced Accuracy: 0.5162030344098163
F1: 0.4763749376387383
LogLoss: 0.8732934475007964

              precision    recall  f1-score   support

           0       0.58      0.64      0.61      1115
           1       0.36      0.05      0.09       858
           2       0.63      0.86      0.73      1780

    accuracy                           0.61      3753
   macro avg       0.53      0.52      0.48      3753
weighted avg       0.56      0.61      0.55      3753



In [104]:
cm = confusion_matrix(y_test, y_test_pred_best)

print("CONFUSION MATRIX -", bestClassifierName)
print(cm)
print()
print("Rows = Actual | Columns = Predicted")
print("Classes: [0=loss, 1=draw, 2=win]")

CONFUSION MATRIX - LogisticRegression
[[ 713   39  363]
 [ 292   44  522]
 [ 215   38 1527]]

Rows = Actual | Columns = Predicted
Classes: [0=loss, 1=draw, 2=win]


In [105]:
if bestClassifierName == "RandomForest":
    importanceDf = (
        pd.DataFrame(
            {
                "feature": featureColumns,
                "importance": bestClassifier.feature_importances_,
            }
        )
        .sort_values("importance", ascending=False)
        .head(15)
        .reset_index(drop=True)
    )
    importanceDf
else:
    coefDf = (
        pd.DataFrame(logModel.coef_.T, index=featureColumns, columns=["coef_loss", "coef_draw", "coef_win"])
        .assign(absImpact=lambda frame: frame.abs().sum(axis=1))
        .sort_values("absImpact", ascending=False)
        .head(15)
        .reset_index(drop=False)
        .rename(columns={"index": "feature"})
    )
    coefDf

## 8. Poisson goal model

In [106]:
poissonFeatureColumns = featureColumns.copy()

poissonScaler = StandardScaler()
X_train_p = poissonScaler.fit_transform(trainDf[poissonFeatureColumns])
X_val_p = poissonScaler.transform(valDf[poissonFeatureColumns])
X_test_p = poissonScaler.transform(testDf[poissonFeatureColumns])

y_home_train = trainDf["homeScore"].to_numpy()
y_away_train = trainDf["awayScore"].to_numpy()

poissonHome = PoissonRegressor(alpha=0.5, max_iter=1000)
poissonAway = PoissonRegressor(alpha=0.5, max_iter=1000)

poissonHome.fit(X_train_p, y_home_train)
poissonAway.fit(X_train_p, trainDf["awayScore"].to_numpy())

lambdaHome_val = np.clip(poissonHome.predict(X_val_p), 0.01, None)
lambdaAway_val = np.clip(poissonAway.predict(X_val_p), 0.01, None)
lambdaHome_test = np.clip(poissonHome.predict(X_test_p), 0.01, None)
lambdaAway_test = np.clip(poissonAway.predict(X_test_p), 0.01, None)

In [107]:
def matchOutcomeProbabilities(lambdaHome, lambdaAway, maxGoals=10):
    goals = np.arange(maxGoals + 1)
    pmfHome = poisson.pmf(goals, lambdaHome)
    pmfAway = poisson.pmf(goals, lambdaAway)
    probabilityGrid = np.outer(pmfHome, pmfAway)

    totalProbability = probabilityGrid.sum()
    if totalProbability > 0:
        probabilityGrid = probabilityGrid / totalProbability

    probWin = np.tril(probabilityGrid, -1).sum()
    probDraw = np.trace(probabilityGrid)
    probLoss = np.triu(probabilityGrid, 1).sum()

    return np.array([probLoss, probDraw, probWin])


def predictPoissonResults(lambdaHomeArray, lambdaAwayArray, maxGoals=10):
    predictedClasses = []
    predictedProbabilities = []

    for lambdaHome, lambdaAway in zip(lambdaHomeArray, lambdaAwayArray):
        probs = matchOutcomeProbabilities(lambdaHome, lambdaAway, maxGoals=maxGoals)
        predictedClasses.append(int(np.argmax(probs)))
        predictedProbabilities.append(probs)

    return np.array(predictedClasses), np.array(predictedProbabilities)


y_val_pred_poisson, y_val_proba_poisson = predictPoissonResults(lambdaHome_val, lambdaAway_val, maxGoals=10)
y_test_pred_poisson, y_test_proba_poisson = predictPoissonResults(lambdaHome_test, lambdaAway_test, maxGoals=10)

poissonValidationResults = pd.DataFrame(
    [metricRow("PoissonGoalModel", y_val, y_val_pred_poisson, y_val_proba_poisson)]
)
poissonTestResults = pd.DataFrame(
    [metricRow("PoissonGoalModel", y_test, y_test_pred_poisson, y_test_proba_poisson)]
)

poissonValidationResults

,model,accuracy,balancedAccuracy,f1,logLoss
0,PoissonGoalModel,0.602773,0.503754,0.442238,0.864455


In [108]:
print("POISSON MODEL (TEST)")
print("Accuracy:", accuracy_score(y_test, y_test_pred_poisson))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_test_pred_poisson))
print("F1:", f1_score(y_test, y_test_pred_poisson, average="macro"))
print("LogLoss:", log_loss(y_test, y_test_proba_poisson, labels=[0, 1, 2]))
print()
print(classification_report(y_test, y_test_pred_poisson, zero_division=0))

POISSON MODEL (TEST)
Accuracy: 0.597388755662137
Balanced Accuracy: 0.5001536756184813
F1: 0.43799506847759123
LogLoss: 0.8785860925109349

              precision    recall  f1-score   support

           0       0.55      0.64      0.59      1115
           1       0.00      0.00      0.00       858
           2       0.62      0.86      0.72      1780

    accuracy                           0.60      3753
   macro avg       0.39      0.50      0.44      3753
weighted avg       0.46      0.60      0.52      3753



In [109]:
comparisonDf = pd.concat(
    [
        validationResults.assign(split="validation"),
        poissonValidationResults.assign(split="validation"),
        testBestResults.assign(split="test"),
        poissonTestResults.assign(split="test"),
    ],
    ignore_index=True,
)

comparisonDf.sort_values(["split", "logLoss", "balancedAccuracy"], ascending=[True, True, False]).reset_index(drop=True)

,model,accuracy,balancedAccuracy,f1,logLoss,split
0,LogisticRegression,0.608580,0.516203,0.476375,0.873293,test
1,PoissonGoalModel,0.597389,0.500154,0.437995,0.878586,test
2,LogisticRegression,0.605705,0.510660,0.470925,0.856007,validation
3,PoissonGoalModel,0.602773,0.503754,0.442238,0.864455,validation
4,RandomForest,0.580912,0.525775,0.518648,0.889813,validation
